# 🏛️ Unsloth Studio on Kaggle (Native Cloudflare Tunnel)

> **Old Money Aesthetic Edition** • Automatic Password Capture & Storage in `/tmp`

---

### 📌 Features:
1. **Automatic Password Extraction**: Continuously scans Unsloth Studio console logs for auto-generated passwords, secrets, tokens, and credentials, displaying them prominently in the UI.
2. **Direct Local Port Verification**: Verifies `127.0.0.1:8889` health locally before tunnel creation to eliminate 502 Bad Gateway errors.
3. **60GB+ Scratch Storage Allocation**: Directs ONLY model downloads (`HF_HOME`, `HF_HUB_CACHE`, `TORCH_HOME`, `UNSLOTH_CACHE_DIR`) to Kaggle's `/tmp` scratch space.
4. **Zero Page Snapping**: In-place DOM updates via `display_handle.update()` for smooth scrolling.
5. **Live Diagnostics Console Logs**: Displays real-time stdout/stderr from `[STUDIO]`, `[TUNNEL]`, and `[SYSTEM]`.
6. **Default Password**: unsloth123.

---

> ⚠️ **IMPORTANT**: Ensure **Internet Access** is enabled in your Kaggle Notebook settings (Right Sidebar -> **Settings** -> **Internet: On**).

In [ ]:
import os
import sys
import time
import re
import html
import shutil
import queue
import threading
import subprocess
import urllib.request
import urllib.error
from IPython.display import HTML, display

# ==============================================================================
# 1. MODEL STORAGE REDIRECT TO KAGGLE /tmp (60GB+ Scratch Space)
# ==============================================================================
STUDIO_PORT = 8889

extra_paths = [os.path.expanduser("~/.local/bin"), "/root/.local/bin", "/usr/local/bin"]
env_path = os.environ.get("PATH", "")
os.environ["PATH"] = ":".join(extra_paths) + ":" + env_path if env_path else ":".join(extra_paths)

# Redirect ONLY Model Downloads and Caches to Kaggle /tmp (60GB+ space)
os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/huggingface/hub"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["UNSLOTH_CACHE_DIR"] = "/tmp/unsloth"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton"

for p in ["/tmp/huggingface", "/tmp/huggingface/hub", "/tmp/torch", "/tmp/unsloth"]:
    os.makedirs(p, exist_ok=True)

# ==============================================================================
# 2. FULL-WIDTH OLD MONEY AESTHETIC UI RENDERER
# ==============================================================================
logs_history = []
captured_passwords = []

def render_old_money_box(status="INITIALIZING", subtext="", url=None, pass_val=None, logs=[]):
    status_color = "#2ECC71" if url else "#D4AF37"
    pulse_css = "animation: pulse_green 2s infinite;" if url else "animation: pulse_gold 1.5s infinite;"
    
    pass_banner = ""
    if pass_val:
        pass_banner = f'''
        <div style="margin-top: 14px; padding: 10px 16px; background: #0F1A12; border: 1px solid #2ECC71; border-radius: 4px; display: inline-block;">
            <span style="font-family: 'Georgia', serif; font-size: 12px; color: #C5A059; text-transform: uppercase; letter-spacing: 1px;">🔑 CAPTURED STUDIO PASSWORD / TOKEN: </span>
            <strong style="font-family: 'Courier New', monospace; font-size: 15px; color: #2ECC71; background: #060E08; padding: 2px 8px; border-radius: 3px; margin-left: 6px;">{html.escape(pass_val)}</strong>
        </div>
        '''
    
    if url:
        url_section = f'''
        <div style="margin: 20px 0 22px 0; padding: 22px 28px; background: #060E08; border: 1px solid #D4AF37; box-shadow: inset 0 0 15px rgba(0,0,0,0.85); text-align: center; border-radius: 4px;">
            <div style="font-family: 'Georgia', serif; font-size: 11px; letter-spacing: 2.5px; text-transform: uppercase; color: #C5A059; margin-bottom: 10px;">
                ❖ SECURE CLOUDFLARE TUNNEL LINK ❖
            </div>
            <div style="margin: 12px 0;">
                <a href="{url}" target="_blank" rel="noopener noreferrer" style="font-family: 'Courier New', monospace; font-size: 19px; font-weight: bold; color: #F5EBE0; text-decoration: none; word-break: break-all; border-bottom: 1px dashed #D4AF37; padding-bottom: 3px;">
                    {url}
                </a>
            </div>
            <div style="margin-top: 18px;">
                <a href="{url}" target="_blank" rel="noopener noreferrer" style="display: inline-block; padding: 13px 36px; background: linear-gradient(135deg, #D4AF37 0%, #AA820A 100%); color: #0B130E; font-family: 'Georgia', serif; font-weight: bold; font-size: 14px; letter-spacing: 2px; text-transform: uppercase; text-decoration: none; border-radius: 3px; box-shadow: 0 4px 22px rgba(212, 175, 55, 0.4);">
                    ✦ Open Unsloth Studio in New Tab ✦
                </a>
            </div>
            {pass_banner}
        </div>
        '''
    else:
        url_section = f'''
        <div style="margin: 20px 0; padding: 20px; background: #060E08; border: 1px dashed #7A6229; text-align: center; border-radius: 4px;">
            <div style="font-family: 'Georgia', serif; font-size: 14px; color: #C5A059; letter-spacing: 1px; font-style: italic;">
                {subtext or 'Configuring environment and establishing secure tunnel...'}
            </div>
            {pass_banner}
        </div>
        '''
    
    log_rows = []
    for t_str, tag, msg, color in logs[-90:]:
        escaped_msg = html.escape(msg)
        log_rows.append(f'<div style="margin-bottom: 3px; line-height: 1.4;"><span style="color: #666; margin-right: 8px;">[{t_str}]</span><span style="color: {color}; font-weight: bold; margin-right: 8px;">[{tag}]</span><span style="color: #DCDCDC;">{escaped_msg}</span></div>')
    
    log_content = "".join(log_rows) if log_rows else '<div style="color: #666; font-style: italic;">Waiting for server output...</div>'
        
    html_card = f'''
    <div style="width: 100%; box-sizing: border-box; padding: 10px 0; font-family: 'Georgia', serif;">
        <style>
            @keyframes pulse_green {{
                0% {{ box-shadow: 0 0 0 0 rgba(46, 204, 113, 0.8); }}
                70% {{ box-shadow: 0 0 0 10px rgba(46, 204, 113, 0); }}
                100% {{ box-shadow: 0 0 0 0 rgba(46, 204, 113, 0); }}
            }}
            @keyframes pulse_gold {{
                0% {{ box-shadow: 0 0 0 0 rgba(212, 175, 55, 0.8); }}
                70% {{ box-shadow: 0 0 0 10px rgba(212, 175, 55, 0); }}
                100% {{ box-shadow: 0 0 0 0 rgba(212, 175, 55, 0); }}
            }}
        </style>
        <div style="width: 100%; box-sizing: border-box; background: linear-gradient(145deg, #101C14 0%, #08100B 100%); border: 2px solid #D4AF37; outline: 1px solid #7A6229; outline-offset: -6px; border-radius: 6px; padding: 28px 32px; box-shadow: 0 16px 50px rgba(0,0,0,0.8); color: #F5EBE0;">
            
            <div style="text-align: center; margin-bottom: 12px;">
                <div style="font-size: 20px; color: #D4AF37; letter-spacing: 6px;">✦ 🏛️ ✦</div>
                <h2 style="font-family: 'Georgia', serif; font-size: 24px; font-weight: 700; color: #F5EBE0; margin: 4px 0 2px 0; letter-spacing: 2.5px; text-transform: uppercase;">
                    UNSLOTH STUDIO
                </h2>
                <div style="font-size: 11px; font-style: italic; color: #C5A059; letter-spacing: 1.5px;">
                    Native Secure Cloudflare Tunnel • High Capacity Model Cache (/tmp)
                </div>
            </div>

            <div style="text-align: center; color: #7A6229; font-size: 10px; margin: 10px 0 18px 0; letter-spacing: 3px;">
                ◆ ────────────────────────────── ❖ ────────────────────────────── ◆
            </div>

            <div style="display: flex; align-items: center; justify-content: space-between; background: #060E08; padding: 12px 20px; border: 1px solid #233627; border-radius: 4px;">
                <div style="display: flex; align-items: center; gap: 12px;">
                    <span style="display: inline-block; width: 11px; height: 11px; background-color: {status_color}; border-radius: 50%; {pulse_css}"></span>
                    <span style="font-family: 'Georgia', serif; font-size: 13px; font-weight: bold; letter-spacing: 1.5px; color: #F5EBE0; text-transform: uppercase;">
                        {status}
                    </span>
                </div>
                <span style="font-family: 'Courier New', monospace; font-size: 11px; color: #C5A059;">
                    PORT: {STUDIO_PORT} • STORAGE: /tmp/huggingface (~60GB+)
                </span>
            </div>

            {url_section}

            <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 14px; margin-bottom: 22px;">
                <div style="background: #050B07; padding: 12px 16px; border: 1px solid #1A2B1E; border-radius: 4px;">
                    <div style="font-size: 10px; color: #C5A059; text-transform: uppercase; letter-spacing: 1px;">💾 Model Storage Directory</div>
                    <div style="font-size: 12px; color: #F5EBE0; font-family: 'Courier New', monospace; margin-top: 4px;">
                        /tmp/huggingface <span style="color:#2ECC71;">(60GB+ Scratch Disk)</span>
                    </div>
                </div>
                <div style="background: #050B07; padding: 12px 16px; border: 1px solid #1A2B1E; border-radius: 4px;">
                    <div style="font-size: 10px; color: #C5A059; text-transform: uppercase; letter-spacing: 1px;">🔒 Encrypted Tunnel</div>
                    <div style="font-size: 12px; color: #F5EBE0; font-family: 'Courier New', monospace; margin-top: 4px;">
                        Cloudflare SSL (HTTPS)
                    </div>
                </div>
            </div>

            <div style="margin-top: 10px;">
                <div style="font-family: 'Georgia', serif; font-size: 12px; font-weight: bold; color: #D4AF37; letter-spacing: 1.5px; text-transform: uppercase; margin-bottom: 8px; display: flex; align-items: center; gap: 8px;">
                    <span>📜</span> LIVE SERVER & TUNNEL CONSOLE LOGS
                </div>
                <div id="console-logs-box" style="height: 250px; overflow-y: auto; background: #040805; border: 1px solid #1E3324; border-radius: 4px; padding: 14px 16px; font-family: 'Courier New', monospace; font-size: 11px; box-shadow: inset 0 0 10px rgba(0,0,0,0.9);">
                    {log_content}
                </div>
            </div>

            <script>
                (function() {{
                    var box = document.getElementById('console-logs-box');
                    if (box) box.scrollTop = box.scrollHeight;
                }})();
            </script>

            <div style="text-align: center; margin-top: 22px; font-size: 10px; color: #586C5E; letter-spacing: 1px;">
                Unsloth AI Studio • Kaggle High-Memory Scratch Storage Active • Process Live
            </div>
        </div>
    </div>
    '''
    return HTML(html_card)

# ==============================================================================
# 3. BACKGROUND THREAD LOG COLLECTOR & HEALTH CHECKERS
# ==============================================================================
log_queue = queue.Queue()

def stream_reader(pipe, prefix, color):
    try:
        for line in iter(pipe.readline, ''):
            if line:
                msg = line.strip()
                if msg:
                    t_str = time.strftime("%H:%M:%S")
                    log_queue.put((t_str, prefix, msg, color))
        pipe.close()
    except Exception:
        pass

def add_sys_log(msg, color="#98C379"):
    t_str = time.strftime("%H:%M:%S")
    log_queue.put((t_str, "SYSTEM", msg, color))

def is_local_server_ready(port=STUDIO_PORT):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{port}", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=2):
            return True
    except urllib.error.HTTPError:
        return True
    except Exception:
        return False

# ==============================================================================
# 4. MAIN LAUNCHER & IN-PLACE DOM UPDATE EXECUTION LOOP
# ==============================================================================
if __name__ == "__main__":
    add_sys_log("Redirecting HuggingFace & PyTorch model downloads to /tmp...")

    display_handle = display(render_old_money_box("INITIALIZING STORAGE", "Configuring model download paths to /tmp (60GB+ space)...", logs=logs_history), display_id=True)
    
    def update_ui_inplace(status, subtext="", url=None, pass_val=None):
        display_handle.update(render_old_money_box(status, subtext, url, pass_val, logs_history))

    cf_bin = "/tmp/cloudflared"
    if not os.path.exists(cf_bin) and not shutil.which("cloudflared"):
        add_sys_log("Downloading Cloudflare Tunnel binary...")
        update_ui_inplace("DOWNLOADING CLOUDFLARED", "Fetching Cloudflare binary...")
        try:
            urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf_bin)
            os.chmod(cf_bin, 0o755)
            add_sys_log("Cloudflared binary saved to /tmp/cloudflared")
        except Exception as e:
            add_sys_log(f"Cloudflared download notice: {e}", color="#E06C75")
    else:
        cf_bin = shutil.which("cloudflared") or "/tmp/cloudflared"
        add_sys_log(f"Using Cloudflared binary at: {cf_bin}")

    add_sys_log("Executing Unsloth installation script...")
    update_ui_inplace("INSTALLING UNSLOTH STUDIO", "Installing Unsloth AI studio CLI...")
    
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", "unsloth"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        subprocess.run("curl -fsSL https://unsloth.ai/install.sh | sh", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        add_sys_log("Unsloth installation completed.")
    except Exception as e:
        add_sys_log(f"Unsloth installer notice: {e}", color="#E06C75")

    unsloth_bin = None
    for candidate in [shutil.which("unsloth"), os.path.expanduser("~/.local/bin/unsloth"), "/root/.local/bin/unsloth", "/usr/local/bin/unsloth", "/opt/conda/bin/unsloth"]:
        if candidate and os.path.exists(candidate):
            unsloth_bin = candidate
            break

    add_sys_log(f"Launching Unsloth Studio on local port {STUDIO_PORT}...")
    update_ui_inplace("LAUNCHING SERVER", f"Starting Unsloth Studio on port {STUDIO_PORT}...")

    if unsloth_bin and os.path.exists(unsloth_bin):
        studio_cmd = [unsloth_bin, "studio", "-H", "0.0.0.0", "-p", str(STUDIO_PORT)]
    else:
        studio_cmd = [sys.executable, "-m", "unsloth", "studio", "-H", "0.0.0.0", "-p", str(STUDIO_PORT)]

    add_sys_log(f"Exec Command: {' '.join(studio_cmd)}", color="#61AFEF")

    studio_proc = subprocess.Popen(studio_cmd, env=os.environ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    
    threading.Thread(target=stream_reader, args=(studio_proc.stdout, "STUDIO", "#61AFEF"), daemon=True).start()

    add_sys_log(f"Verifying local HTTP server health on port {STUDIO_PORT}...")
    server_ready = False
    for attempt in range(20):
        time.sleep(1.0)
        
        if studio_proc.poll() is not None:
            add_sys_log(f"ERROR: Unsloth Studio process exited prematurely with code {studio_proc.returncode}", color="#E06C75")
            break

        if is_local_server_ready(STUDIO_PORT):
            server_ready = True
            add_sys_log(f"SUCCESS: Unsloth Studio web server active on port {STUDIO_PORT}!", color="#2ECC71")
            break

    add_sys_log(f"Establishing Cloudflare Tunnel to http://127.0.0.1:{STUDIO_PORT}...")
    update_ui_inplace("ESTABLISHING TUNNEL", "Requesting Cloudflare HTTPS Tunnel URL...")

    tunnel_cmd = [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{STUDIO_PORT}"]
    tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    threading.Thread(target=stream_reader, args=(tunnel_proc.stdout, "TUNNEL", "#E5C07B"), daemon=True).start()

    tunnel_url = None
    captured_pass = None
    last_ui_update = 0

    add_sys_log("Continuous execution loop active. Server & Tunnel online.")

    try:
        while True:
            time.sleep(0.3)
            now = time.time()
            
            has_new_log = False
            while not log_queue.empty():
                t_str, tag, msg, color = log_queue.get_nowait()
                logs_history.append((t_str, tag, msg, color))
                has_new_log = True
                
                # 1. Capture Cloudflare Tunnel URL
                if not tunnel_url:
                    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", msg)
                    if match:
                        candidate = match.group(0)
                        if "precheck" not in msg and "run_id=" not in msg and "target=" not in msg:
                            tunnel_url = candidate
                            add_sys_log(f"Cloudflare HTTPS Tunnel URL captured: {tunnel_url}", color="#2ECC71")
                
                # 2. Capture auto-generated password / token / secret from log output
                if not captured_pass:
                    pass_match = re.search(r"(?:password|token|key|secret)[:\s=]+([a-zA-Z0-9_-]{4,64})", msg, re.IGNORECASE)
                    if pass_match:
                        candidate_pass = pass_match.group(1)
                        if candidate_pass.lower() not in ["true", "false", "none", "null", "info", "error", "warn"]:
                            captured_pass = candidate_pass
                            add_sys_log(f"CAPTURED AUTO-GENERATED STUDIO PASSWORD: {captured_pass}", color="#2ECC71")
                
                if len(logs_history) > 200:
                    logs_history.pop(0)

            if (has_new_log and (now - last_ui_update > 0.8)) or (now - last_ui_update > 3.0):
                status = "UNSLOTH STUDIO IS LIVE" if tunnel_url else "ESTABLISHING TUNNEL"
                subtext = "" if tunnel_url else "Generating Cloudflare HTTPS tunnel... (Ensure Internet is ON)"
                
                update_ui_inplace(status, subtext=subtext, url=tunnel_url, pass_val=captured_pass)
                last_ui_update = now

    except KeyboardInterrupt:
        add_sys_log("Server execution interrupted by user.", color="#E06C75")
        update_ui_inplace("SERVER STOPPED", url=None)
        studio_proc.terminate()
        tunnel_proc.terminate()
